# Day 03 · 函数、类型注解与模块

> 今天的目标是把“能写语句”推进到“能设计可复用函数”。后面的自动化脚本、数据分析清洗函数、FastAPI 路由、AI/LLM 数据管线，都依赖清晰的函数边界。

## 1. 函数边界：参数进，返回值出

- Python 函数用 `def` 定义，缩进表示函数体。
- 学习时最重要的习惯：函数负责返回结果，调用者决定是否 `print`。
- 对比 C：不需要声明返回类型；对比 JS：函数也是对象，可以赋值和传递。
- 设计函数时先问：输入是什么、输出是什么、是否修改传入对象。

In [ ]:
def greet(name):
    return f"Hello, {name}!"

message = greet("Python")
print(message)

## 2. 参数：位置参数、关键字参数、默认值

- 普通调用：`func(1, 2)`；关键字调用：`func(x=1, y=2)`。
- 有默认值的参数适合表达“可选配置”：`tax_rate=0.0`。
- 默认参数只在函数定义时创建一次。不要写 `items=[]` 这类可变默认值。
- 如果参数很多，优先用关键字调用提升可读性。

In [ ]:
def calculate_total(subtotal, tax_rate=0.0, discount=0.0):
    return subtotal * (1 - discount) * (1 + tax_rate)

print(calculate_total(100, tax_rate=0.08, discount=0.1))

## 3. 类型注解：给人和工具看的契约

- 写法：`def add(a: int, b: int) -> int:`。
- `list[str]`、`dict[str, int]`、`str | None` 是常见注解。
- 类型注解默认不强制运行时检查，但能帮助编辑器、mypy、读代码的人。
- 先把输入输出标清楚，比追求复杂类型更重要。

In [ ]:
def safe_head(items: list[str]) -> str | None:
    if not items:
        return None
    return items[0]

print(safe_head(["a", "b"]))
print(safe_head([]))

## 4. `None` 与早返回

- `None` 表示没有值，类似 JS 的 `null`，但判断时用 `is None`。
- 空输入、找不到结果、没有可计算值时，可以返回 `None`。
- guard clause：先处理特殊情况，再写主逻辑，减少嵌套。

In [ ]:
def average(numbers: list[float]) -> float | None:
    if not numbers:
        return None
    return sum(numbers) / len(numbers)

print(average([]))
print(average([1, 2, 3]))

## 5. `*args`、`**kwargs` 与 keyword-only

- `*args` 收集多余位置参数，得到 tuple。
- `**kwargs` 收集多余关键字参数，得到 dict。
- 函数签名里的单独 `*` 表示后面的参数必须用关键字传入：`def f(x, *, normalize=True)`。
- 这些能力很灵活，但不要滥用；清晰签名优先。

In [ ]:
def make_profile(name: str, *, active=True, **extra):
    profile = {"name": name, "active": active}
    profile.update(extra)
    return profile

print(make_profile("Alice", role="admin"))

## 6. 可变对象与副作用

- `list`、`dict` 是可变对象，函数里修改它会影响调用者手里的对象。
- 如果函数目标是“计算新结果”，通常先复制再改。
- 可变默认参数坑：`def add(x, items=[])` 会让多次调用共享同一个列表。
- 安全写法：默认用 `None`，函数内部再创建新列表。

In [ ]:
def add_item(items: list[str] | None, item: str) -> list[str]:
    result = [] if items is None else items.copy()
    result.append(item)
    return result

original = ["a"]
print(add_item(original, "b"))
print(original)

## 7. 模块与导入

- 每个 `.py` 文件就是一个模块。
- `import module` 导入模块；`from module import name` 导入名字。
- 练习里的 `tests/` 通过 `from exercises.xxx import func` 导入你的函数。
- 一个模块可以暴露多个小函数。主函数调用 helper 函数，是组织代码的第一步。

In [ ]:
def normalize_name(name: str) -> str:
    return name.strip().title()

def normalize_names(names: list[str]) -> list[str]:
    return [normalize_name(name) for name in names]

print(normalize_names([" alice ", "BOB"]))

## 8. 闭包（Closure）

- 函数里**定义并返回另一个函数**，内层函数记住外层的变量——这就是闭包。
- Python 按**引用**捕获外层变量，不是拷贝，而且是**延迟绑定**（用的时候才取值）。
  - 经典坑：`funcs = [lambda: i for i in range(3)]`，三个函数全返回 2，因为共享同一个 `i`。
  - 解法：用默认参数当场固定，`lambda i=i: i`。
- 要在内层**修改**外层变量，得声明 `nonlocal`（类比修改全局变量要 `global`）。
- 对比 C：没有指针，靠 `nonlocal` 表达"改外层那个名字"。

In [ ]:
# 闭包：内层函数记住外层变量
def make_adder(n):
    def add(x):
        return x + n
    return add

add5 = make_adder(5)
print(add5(10))  # 15

In [ ]:
# 延迟绑定的坑 + 修复
bad = [lambda: i for i in range(3)]
print([f() for f in bad])        # [2, 2, 2]
good = [lambda i=i: i for i in range(3)]
print([f() for f in good])       # [0, 1, 2]

## 9. 装饰器（Decorator）

- 装饰器就是"**接收一个函数、返回一个新函数**"的函数。`@deco` 写在 def 上方，等价于 `f = deco(f)`。
- 典型写法：内层 `wrapper(*args, **kwargs)` 包住原函数，前后加逻辑（计时、计数、缓存、鉴权）。
- **务必用 `functools.wraps`**：否则被装饰后函数的 `__name__`/`__doc__` 会变成 `wrapper`，调试和文档全乱。
- 对比 JS：类似高阶函数包装；Python 的 `@` 语法糖让它成为一等写法。

In [ ]:
# 装饰器 + functools.wraps
import functools

def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"调用 {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_calls
def greet(name):
    """打招呼。"""
    return f"hi {name}"

print(greet("world"))
print(greet.__name__)   # greet（被 wraps 保留），不是 wrapper

## 10. 今日练习（10 题，难度递增）

1. `calculate_total`：默认参数、返回值、四舍五入。
2. `build_user_profile`：`**kwargs` 与可选字段。
3. `safe_average`：空输入返回 `None`，练习 `float | None`。
4. `parse_tags`：keyword-only 参数、字符串清洗、保序去重。
5. `format_report`：`*args` 与 keyword-only 开关。
6. `add_task`：用 `None` 做安全默认参数，避免可变默认参数和意外修改输入。
7. `summarize_people`：一个模块内多个函数协作，模拟小型数据处理模块。
8. `make_counter`：闭包 + `nonlocal` 实现计数器。
9. `count_calls`：装饰器记录调用次数，`functools.wraps` 保留元数据。
10. `memoize`：用字典做缓存的装饰器。

完成后进入本目录运行 `pytest -q` 验证。预计耗时：讲解 35-45 分钟，练习 75-90 分钟；内容偏多，可分两次学（先 ex1–ex7，再闭包/装饰器 ex8–ex10）。